# Pandas vs DuckDB y CSV vs Parquet

**John González**

*28 de septiembre de 2026*

**Objetivo**: Ver las diferencias entre estos motores analíticos y formatos de datos.

**Recursos**: Este espacio de trabajo

## Recursos máquina

Antes de iniciar verifiquemos el tamaño de RAM y DISCO DURO de la máquina

In [ ]:
! free -h  # RAM

               total        used        free      shared  buff/cache   available
Mem:           7.8Gi       3.1Gi       456Mi        66Mi       4.5Gi       4.6Gi
Swap:             0B          0B          0B


In [ ]:
! df -h  # Disk Space

Filesystem      Size  Used Avail Use% Mounted on
overlay          32G  7.6G   23G  26% /
tmpfs            64M     0   64M   0% /dev
shm              64M     0   64M   0% /dev/shm
/dev/sdb1        44G  1.6G   41G   4% /tmp
/dev/root        29G   15G   15G  51% /vscode
/dev/loop4       32G  7.6G   23G  26% /workspaces
tmpfs           4.0K     0  4.0K   0% /proc/acpi


In [ ]:
! pwd # Print Working Directory

/workspaces/bigdata/notebooks


## Descarga de datos

VAmos a descargar los datos de predicción de tarifa de taxis de Nueva York desde [Kaggle](https://www.kaggle.com/competitions/new-york-city-taxi-fare-prediction/data) 

In [10]:
import kagglehub
from dotenv import load_dotenv

load_dotenv("../.env")

# Download latest version
path = kagglehub.competition_download('new-york-city-taxi-fare-prediction')

print("Path to competition files:", path)

100%|██████████| 1.56G/1.56G [00:26<00:00, 63.8MB/s]


Extracting files...
Path to competition files: /home/vscode/.cache/kagglehub/competitions/new-york-city-taxi-fare-prediction


In [ ]:
! ls -lah $path # Verficamos los archivos descargados

total 5.4G
drwxr-xr-x 2 vscode vscode 4.0K Sep 29 02:19 .
drwxr-xr-x 3 vscode vscode 4.0K Sep 29 02:22 ..
-rw-r--r-- 1 vscode vscode  486 Sep 29 02:19 GCP-Coupons-Instructions.rtf
-rw-r--r-- 1 vscode vscode 336K Sep 29 02:19 sample_submission.csv
-rw-r--r-- 1 vscode vscode 960K Sep 29 02:19 test.csv
-rw-r--r-- 1 vscode vscode 5.4G Sep 29 02:22 train.csv


## Lectura con Pandas

In [22]:
import pandas as pd
import time

print("Iniciando lectura pandas")

try:
    start_time = time.time()
    df = pd.read_csv(f"{path}/train.csv")
    media = df['fare_amount'].mean()
    end_time = time.time()
    print(f"Tiempo de lectura pandas: {end_time - start_time:.2f} segundos")
except Exception as e:
    print(f"Error al leer el archivo: {e}")

Iniciando lectura pandas


: 

## Leyendo con DuckDB

DuckDB optimiza la RAM (Out-of-Core), toma los datos y va cargando lo que vaya necesitando en la RAM, aprovechando los recursos disponibles de un solo nodo.



In [4]:
import duckdb
import time

path = "/home/vscode/.cache/kagglehub/competitions/new-york-city-taxi-fare-prediction"

print("Iniciando lectura DuckDB")

try:
    start_time = time.time()
    result = duckdb.execute(f"SELECT * FROM '{path}/train.csv'").fetchall()
    end_time = time.time()
    print(f"Tiempo de lectura DuckDB: {end_time - start_time:.2f} segundos")
except Exception as e:
    print(f"Error al leer el archivo: {e}")

Iniciando lectura DuckDB


: 

In [1]:
import duckdb
import time

path = "/home/vscode/.cache/kagglehub/competitions/new-york-city-taxi-fare-prediction"

print("Iniciando lectura DuckDB")

try:
    start_time = time.time()
    result = duckdb.execute(f"SELECT AVG(fare_amount) FROM '{path}/train.csv'").fetchall()
    media = result[0][0]
    print(f"Media del fare_amount: {media}")
    end_time = time.time()
    print(f"Tiempo de lectura DuckDB: {end_time - start_time:.2f} segundos")
except Exception as e:
    print(f"Error al leer el archivo: {e}")

Iniciando lectura DuckDB
Media del fare_amount: 11.345045601713737
Tiempo de lectura DuckDB: 34.43 segundos


### Ejercicio

1. Explique porque pandas falla
2. Porque la primera ejecución de duckdb falla
3. Porque la segunda ejecución funciona

NOTA: Con sus propias palabras